In [ ]:
#base_model
#draft_model
# 
# 
# Implement speculative decoding technique 
# 1. token speculation 
# 2. parallel verification 
# 3. rejection sampling
import numpy as np
import copy

def softmax(x, axis=-1):
    """Stable softmax; works on NumPy versions before np.softmax existed (NumPy 2.0+)."""
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

B, T, C = 1, 10, 10
VOCAB_SIZE = 60000
EOS_TOKEN = 101
NUM_SAMPLES = 10



def draft_model(input_idx):
    # we only only sinmulating the logits for the current token
    logits = np.random.randn(B, T, vocab_size)
    logits = softmax(logits, axis=-1)
    return logits

def base_model(input_idx):
    logits = np.random.randn(B, T, vocab_size)
    logits = softmax(logits, axis=-1)
    return logits

input_idx = np.random.randint(0, 100, (1, T)) # [1, T]


#speculative decoding
def speculative_decoding(input_idx):
    # assume draft_model return probs of shape [B, T, V]
    # assume base_model return probs of shape [B, T, V]

    temp_seq = input_idx.copy()

    draft_tokens = []
    draft_probs_list = []

    for _ in range(NUM_SAMPLES):
        q_probs = draft_model(temp_seq)[0, -1, :]

        # sample a token from the draft model
        draft_tok  = np.argmax(q_probs)
        draft_tokens.append(draft_tok)
        draft_probs_list.append(q_probs)

        temp_seq = np.concatenate([temp_seq, draft_tok], axis=-1)

    draft_probs = np.array(draft_probs_list)
    draft_tokens = np.array(temp_seq[0, input_idx.shape[1]:])


    # verification step 
    base_probs  = base_model(np.concatenate([input_idx, draft_tokens], axis=-1))
    base_probs = base_probs[0, -(NUM_SAMPLES+1):, :]

    n_accepted = 0
    for i in range(NUM_SAMPLES):
        q_p = draft_probs[i]
        b_p = base_probs[i]

        drafted_token = draft_tokens[i]

        p_accept = min(1, (b_p[drafted_token] / q_p[drafted_token]))
        
        if np.random.rand() < p_accept:
            n_accepted += 1
            
            current_seq = np.concatenate([current_seq, drafted_token], axis=-1)

            if drafted_token == eos_token:
                return current_seq

        else:
            break

        
